# Command Design Pattern 

explained using the classic Smart Home Remote example.

#### The Concept

The Command Pattern encapsulates a request as an object, allowing you to parameterize clients with different requests, queue or log requests, and support undoable operations. Analogy: Ordering at a Restaurant.
- **You (Client)**: Give an order to the waiter.
- **Waiter (Invoker)**: Takes the order (Command) and places it on the counter. He doesn't cook the food; he just invokes the command.
- **Chef (Receiver)**: Actually cooks the food when the order comes in.

## The Classic OOP Way (Java-Style)

In this strict approach, everything is a class. We need an Interface for the Command, Concrete Classes for every action (`LightOn`, `LightOff`, `FanOn`, `FanOff`), and an Invoker to hold them.

#### THE RECEIVER (The Smart Device)

In [1]:
class Light:
    def turn_on(self):
        print("Light: The light is ON")

    def turn_off(self):
        print("Light: The light is OFF")

#### THE COMMAND INTERFACE

In [2]:
from abc import ABC, abstractmethod

class Command(ABC):
    @abstractmethod
    def execute(self) -> None:
        pass

    @abstractmethod
    def undo(self) -> None:
        pass

#### CONCRETE COMMANDS

In [3]:
class LightOnCommand(Command):
    def __init__(self, light: Light):
        self.light = light

    def execute(self) -> None:
        self.light.turn_on()

    def undo(self) -> None:
        self.light.turn_off()

class LightOffCommand(Command):
    def __init__(self, light: Light):
        self.light = light

    def execute(self) -> None:
        self.light.turn_off()

    def undo(self) -> None:
        self.light.turn_on()

#### THE INVOKER (The Remote Control)

In [4]:
from typing import List

class RemoteControl:
    def __init__(self):
        self.history: List[Command] = []

    def press_button(self, command: Command):
        print("[Remote] Button pressed.")
        command.execute()
        self.history.append(command)

    def press_undo(self):
        if not self.history:
            print("[Remote] Nothing to undo.")
            return
        
        print("[Remote] Undo pressed.")
        last_command = self.history.pop()
        last_command.undo()

#### CLIENT CODE

In [5]:
def main():
    # Setup
    living_room_light = Light()
    
    # Create concrete commands
    turn_on = LightOnCommand(living_room_light)
    turn_off = LightOffCommand(living_room_light)
    
    # Invoker
    remote = RemoteControl()

    # Execute
    remote.press_button(turn_on)  # Light ON
    remote.press_button(turn_off) # Light OFF
    
    # Undo
    remote.press_undo() # Undo OFF -> Turns ON

if __name__ == "__main__":
    main()

[Remote] Button pressed.
Light: The light is ON
[Remote] Button pressed.
Light: The light is OFF
[Remote] Undo pressed.
Light: The light is ON


## The Pythonic Way

In Python, functions are objects. We don't need to create a `LightOnCommand` class just to call `light.turn_on()`. We can pass the method itself! For "Undo" functionality, we can store a tuple of `(execute_func, undo_func)`.

#### THE RECEIVER (Same as before)

In [6]:
class Light:
    def on(self): print("💡 Light ON")
    def off(self): print("🌑 Light OFF")

#### THE INVOKER (Stores Functions)

In [9]:
from typing import Callable, List, Tuple

class SmartRemote:
    def __init__(self):
        # History stores tuples: (do_function, undo_function)
        self.history: List[Tuple[Callable, Callable]] = []

    def execute(self, do_command: Callable, undo_command: Callable):
        print("[Remote] Executing...")
        do_command()
        # Save the pair so we know how to undo this specific action later
        self.history.append((do_command, undo_command))

    def undo(self):
        if not self.history:
            print("[Remote] Nothing to undo.")
            return
        
        print("[Remote] Undoing...")
        # Unpack the last tuple
        _, undo_func = self.history.pop()
        undo_func()

#### CLIENT CODE

In [10]:
def main():
    light = Light()
    remote = SmartRemote()

    # Look how clean this is! 
    # No "LightOnCommand" classes. Just passing methods directly.
    
    # 1. Turn Light ON (Undo is OFF)
    remote.execute(light.on, light.off)

    # 2. Turn Light OFF (Undo is ON)
    remote.execute(light.off, light.on)

    print("-" * 20)

    # 3. Undo last action
    remote.undo() # Should turn Light ON

if __name__ == "__main__":
    main()

[Remote] Executing...
💡 Light ON
[Remote] Executing...
🌑 Light OFF
--------------------
[Remote] Undoing...
💡 Light ON


### Key Differences

| Feature          | Classic OOP                                                     | Pythonic                                                         |
|------------------|-----------------------------------------------------------------|------------------------------------------------------------------|
| **Command Object** | A Class (`LightOnCommand`) implementing an Interface.           | A Function/Method (`light.on`) or any callable.                  |
| **Boilerplate**    | High — one class per action.                                    | Zero — uses existing methods.                                    |
| **State**          | Can store complex state inside the Command object.              | Uses closures or `functools.partial` if parameters are needed.   |


#### When to use which?

- **Use the Class approach** if the command has a complex lifecycle, needs to serialize itself to a database, or holds significant state (e.g., a text editor command that remembers the deleted text).
- **Use the Pythonic approach** for GUI buttons, simple menu actions, or simple callback queues.

# Command Design Pattern 

explained with a complex, real-world example: A Text Editor with Undo/Redo

#### The Scenario: Undoable Text Operations

In a text editor, simply performing an action (like typing "Hello") isn't enough. You need to remember what was typed and where, so that if the user hits Ctrl+Z, you can remove exactly those characters from that specific position.
- **Copy/Paste**: Needs to access the clipboard.
- **Type/Delete**: Needs to modify the document state.
- **Undo/Redo**: Needs a history stack.

## The Classic OOP Way (Java-Style)

We create an abstract `Command` class. Each operation (`WriteCommand`, `DeleteCommand`) saves the state before the execution so it can restore it later.

#### RECEIVER (The Document)

In [11]:
class Document:
    def __init__(self):
        self._content = ""

    def insert(self, text: str, position: int):
        self._content = self._content[:position] + text + self._content[position:]
        print(f"[Doc] Inserted '{text}' at {position}. Result: \"{self._content}\"")

    def delete(self, length: int, position: int) -> str:
        # Save deleted text to return it (needed for undo)
        deleted_text = self._content[position : position + length]
        self._content = self._content[:position] + self._content[position + length:]
        print(f"[Doc] Deleted {length} chars at {position}. Result: \"{self._content}\"")
        return deleted_text

    def get_content(self):
        return self._content

#### COMMAND INTERFACE

In [13]:
from abc import ABC, abstractmethod

class Command(ABC):
    @abstractmethod
    def execute(self) -> None: pass
    
    @abstractmethod
    def undo(self) -> None: pass

#### CONCRETE COMMANDS (Stateful)

In [15]:
class WriteCommand(Command):
    """
    Writes text. Undo deletes it.
    """
    def __init__(self, doc: Document, text: str, position: int):
        self.doc = doc
        self.text = text
        self.position = position

    def execute(self) -> None:
        self.doc.insert(self.text, self.position)

    def undo(self) -> None:
        # To undo a write, we delete what was written
        self.doc.delete(len(self.text), self.position)

class DeleteCommand(Command):
    """
    Deletes text. Undo re-inserts it.
    """
    def __init__(self, doc: Document, length: int, position: int):
        self.doc = doc
        self.length = length
        self.position = position
        self.deleted_text = "" # State backup

    def execute(self) -> None:
        # We MUST save the deleted text to undo this later
        self.deleted_text = self.doc.delete(self.length, self.position)

    def undo(self) -> None:
        # Restore the deleted text
        self.doc.insert(self.deleted_text, self.position)

#### INVOKER (The Editor Controller)

In [17]:
from typing import List

class TextEditor:
    def __init__(self):
        self.history: List[Command] = []
        self.redo_stack: List[Command] = []

    def execute_command(self, cmd: Command):
        cmd.execute()
        self.history.append(cmd)
        self.redo_stack.clear() # New action clears redo history

    def undo(self):
        if not self.history:
            print("Nothing to undo.")
            return
        cmd = self.history.pop()
        cmd.undo()
        self.redo_stack.append(cmd)

    def redo(self):
        if not self.redo_stack:
            print("Nothing to redo.")
            return
        cmd = self.redo_stack.pop()
        cmd.execute()
        self.history.append(cmd)

#### CLIENT CODE

In [21]:
def main():
    doc = Document()
    editor = TextEditor()

    print("--- 1. Writing 'Hello ' ---")
    cmd1 = WriteCommand(doc, "Hello ", 0)
    editor.execute_command(cmd1)

    print("\n--- 2. Writing 'World' ---")
    cmd2 = WriteCommand(doc, "World", 6)
    editor.execute_command(cmd2)

    print("\n--- 3. Undo (Removes 'World') ---")
    editor.undo()

    print("\n--- 4. Redo (Restores 'World') ---")
    editor.redo()
    
    print("\n--- 5. Delete 'Hello ' ---")
    cmd3 = DeleteCommand(doc, 6, 0)
    editor.execute_command(cmd3)

    print("\n--- 6. Undo Delete (Restores 'Hello ') ---")
    editor.undo()

    print("\n--- Document content ---")
    print(doc.get_content())

if __name__ == "__main__":
    main()

--- 1. Writing 'Hello ' ---
[Doc] Inserted 'Hello ' at 0. Result: "Hello "

--- 2. Writing 'World' ---
[Doc] Inserted 'World' at 6. Result: "Hello World"

--- 3. Undo (Removes 'World') ---
[Doc] Deleted 5 chars at 6. Result: "Hello "

--- 4. Redo (Restores 'World') ---
[Doc] Inserted 'World' at 6. Result: "Hello World"

--- 5. Delete 'Hello ' ---
[Doc] Deleted 6 chars at 0. Result: "World"

--- 6. Undo Delete (Restores 'Hello ') ---
[Doc] Inserted 'Hello ' at 0. Result: "Hello World"

--- Document content ---
Hello World


## The Pythonic Way

We can use **Functions** and **Closures** to capture state. Instead of a `DeleteCommand` class storing `self.deleted_text`, we can create a closure that "**remembers"** the deleted text automatically.

We can also represent the history as a list of **"Undo Functions"**.

#### RECEIVER (The Document)

In [22]:
from dataclasses import dataclass

@dataclass
class Document:
    content: str = ""

    def insert(self, text, pos):
        self.content = self.content[:pos] + text + self.content[pos:]
        print(f"[Doc] State: \"{self.content}\"")

    def delete(self, length, pos):
        deleted = self.content[pos : pos + length]
        self.content = self.content[:pos] + self.content[pos + length:]
        print(f"[Doc] State: \"{self.content}\"")
        return deleted

#### INVOKER (The Command Manager)

In [23]:
from typing import List, Callable 

class CommandManager:
    def __init__(self):
        # Stack of Undo functions
        self.undo_stack: List[Callable] = []
        # Stack of Redo functions (execute logic)
        self.redo_stack: List[Callable] = []

    def execute(self, do_func: Callable, undo_func: Callable):
        do_func()
        # Push the UNDO logic onto the stack
        self.undo_stack.append(undo_func)
        # Clear redo on new action
        self.redo_stack.clear() 

    def undo(self):
        if not self.undo_stack: return
        print("-> UNDO")
        undo_action = self.undo_stack.pop()
        undo_action()

#### CLIENT CODE (Using Closures)

In [24]:
def main():
    doc = Document()
    manager = CommandManager()

    # --- ACTION 1: WRITE "PYTHON" ---
    print("Action 1: Type 'Python'")
    
    # We define the logic right here
    def do_write(): doc.insert("Python", 0)
    def undo_write(): doc.delete(6, 0)

    manager.execute(do_write, undo_write)


    # --- ACTION 2: DELETE "PY" ---
    print("\nAction 2: Delete 'Py'")
    
    # Capture state before deletion!
    # Note: Real-world text editors usually calculate this inside the execute block,
    # but for simplicity, we simulate knowing what to delete.
    deleted_text = "Py" 
    
    def do_delete(): doc.delete(2, 0)
    def undo_delete(): doc.insert(deleted_text, 0) # Closure captures 'deleted_text'

    manager.execute(do_delete, undo_delete)


    # --- UNDO OPERATIONS ---
    print("\n--- Undo Sequence ---")
    manager.undo() # Should restore "Py" -> "Python"
    manager.undo() # Should delete "Python" -> ""

if __name__ == "__main__":
    main()

Action 1: Type 'Python'
[Doc] State: "Python"

Action 2: Delete 'Py'
[Doc] State: "thon"

--- Undo Sequence ---
-> UNDO
[Doc] State: "Python"
-> UNDO
[Doc] State: ""


#### Why Closures work well here

In the Pythonic version, `undo_delete` captures the variable `deleted_text` from the local scope. We didn't need to create a class `DeleteCommand` just to hold that one string variable.

#### Summary
- **Java Way**: Uses explicit Command objects. Great if you need to serialize commands (save history to disk) or if the logic is very heavy.
- **Python Way**: Uses functions and closures. Great for in-memory undo stacks and cleaner UI event handling.